# Holo-GNN: V4.0 Final Training & Evaluation

**Architecture:** ESM-2 (t6 8M) backbone · Dual-Track Attention Graph (V2.0) · Mechanistic Feature Injection (V3.0) · Siamese Antisymmetric Pass (V4.0)

This notebook executes the complete V4.0 pipeline in three sequential phases:

- **Phase 1 — Training:** 5-epoch Siamese training loop on MegaScale cDNA stability data using `AntisymmetricLoss`, logging both `Fidelity` and `Antisymmetry` loss components per epoch.
- **Phase 2 — Benchmarking:** Full test-set evaluation computing RMSE and Pearson *r* between predicted and experimental ΔΔG values.
- **Phase 3 — Visualisation:** Matplotlib figures for training loss curves, scatter plot (predicted vs. experimental ΔΔG), and antisymmetry convergence — saved to `holognn_final_metrics.png`.

---
> **Vertex AI target:** NVIDIA L4 GPU (24 GB VRAM) · 8 vCPU DataLoader workers  
> **Run:** `Runtime → Run all`

In [ ]:
# ── Cell 2: Install required domain-specific libraries for Vertex AI environment ──
!pip install "numpy<2" "transformers==4.33.0" biopython

# torch_geometric — required for GATConv message-passing layers
import subprocess, sys
try:
    import torch_geometric
    print(f"torch_geometric {torch_geometric.__version__} already present.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    print("torch_geometric installed.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Data Path Variables · Dataset Initialisation · DataLoader Setup
# ══════════════════════════════════════════════════════════════════════════════

import os, math, time, random
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from tqdm.notebook import tqdm
from transformers import EsmTokenizer, EsmModel
from Bio.Seq import Seq

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ── PyG (optional — enables GATConv) ─────────────────────────────────────────
try:
    from torch_geometric.nn import GATConv
    _PYGEO = True
    print("✅ GATConv available — GNN layers ENABLED")
except ImportError:
    GATConv = None
    _PYGEO = False
    print("⚠️  torch_geometric missing — ESM-2 mean-pool fallback active")

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   Device : {device}")

# ────────────────────────────────────────────────────────────────────────────
# DATA PATH VARIABLES  ← edit here
# ────────────────────────────────────────────────────────────────────────────
DATA_PATH = (
    "data/mega_scale_cdna/Processed_K50_dG_datasets/"
    "Processed_K50_dG_datasets/Tsuboyama2023_Dataset1_20230416.csv"
)
CHECKPOINT_PATH = "holognn_v4_final.pth"

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_SAMPLES    = 100_000
MAX_SEQ_LEN    = 100
BATCH_SIZE     = 64      # L4 24 GB VRAM
NUM_WORKERS    = 8
EPOCHS         = 5
LEARNING_RATE  = 1e-4
ALPHA_ASYM     = 1.0     # AntisymmetricLoss antisymmetry weight
VAL_SPLIT      = 0.10
TEST_SPLIT     = 0.10

# ── Amino acid charge table (used by mechanistic feature generator) ───────────
_AA_CHARGE = {'R': +1, 'K': +1, 'H': +1, 'D': -1, 'E': -1}

# ── Kyte-Doolittle hydrophobicity scale (CAI proxy) ───────────────────────────
_KD = {
    'I': 4.5, 'V': 4.2, 'L': 3.8, 'F': 2.8, 'C': 2.5, 'M': 1.9, 'A': 1.8,
    'G': -0.4, 'T': -0.7, 'S': -0.8, 'W': -0.9, 'Y': -1.3, 'P': -1.6,
    'H': -3.2, 'E': -3.5, 'Q': -3.5, 'D': -3.5, 'N': -3.5, 'K': -3.9, 'R': -4.5,
}
_KD_MIN, _KD_MAX = -4.5, 4.5


# ════════════════════════════════════════════════════════════
# V3.0  Mechanistic Feature Generator  (from src/dataset.py)
# ════════════════════════════════════════════════════════════
def _mechanistic_features(protein_seq: str, max_length: int) -> torch.Tensor:
    """
    Returns (max_length, 3) float32 tensor of MinMax-scaled [0,1] features:
      ch0 — mRNA_fold   : local GC-content proxy (window ±1 residue)
      ch1 — CAI         : Kyte-Doolittle hydrophobicity proxy, scaled to [0,1]
      ch2 — Charge      : sliding-window (±2) net charge, scaled to [0,1]
    Padding positions remain zero.
    """
    GC_RICH = {'G', 'A', 'P', 'R', 'W', 'C', 'S'}
    L   = min(len(protein_seq), max_length)
    out = torch.zeros(max_length, 3, dtype=torch.float32)
    for i in range(L):
        aa = protein_seq[i].upper()
        window = protein_seq[max(0, i-1): i+2]
        mrna   = sum(1 for a in window if a.upper() in GC_RICH) / max(len(window), 1)
        cai    = (_KD.get(aa, 0.0) - _KD_MIN) / (_KD_MAX - _KD_MIN)
        raw_ch = sum(_AA_CHARGE.get(a.upper(), 0) for a in protein_seq[max(0,i-2):i+3])
        charge = max(0.0, min(1.0, (raw_ch + 5) / 10.0))
        out[i] = torch.tensor([mrna, cai, charge])
    return out


# ════════════════════════════════════════════════
# V3.0  MegaScaleDataset  (from src/dataset.py)
# ════════════════════════════════════════════════
class MegaScaleDataset(Dataset):
    """
    Thermodynamic stability dataset.
    Each __getitem__ returns a dict with:
      input_ids, attention_mask, label, mechanistic_features
    """
    def __init__(self, csv_path: str, max_length: int = MAX_SEQ_LEN):
        self.tok = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
        self.max_length = max_length
        print(f"Loading MegaScale data from {csv_path} ...")
        df       = pd.read_csv(csv_path)
        self.df  = df.dropna(subset=["dna_seq", "deltaG"]).reset_index(drop=True)
        print(f"  {len(self.df):,} valid samples loaded.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        seq     = str(Seq(row["dna_seq"]).translate(to_stop=True))
        label   = float(row["deltaG"])
        enc     = self.tok(
            seq, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids":            enc["input_ids"].squeeze(0),
            "attention_mask":       enc["attention_mask"].squeeze(0),
            "label":                torch.tensor(label, dtype=torch.float),
            "mechanistic_features": _mechanistic_features(seq, self.max_length),
        }


# ── Build dataset & splits ────────────────────────────────────────────────────
full_ds   = MegaScaleDataset(DATA_PATH)
n         = min(MAX_SAMPLES, len(full_ds))
active_ds = Subset(full_ds, range(n))

n_test  = int(TEST_SPLIT * n)
n_val   = int(VAL_SPLIT  * n)
n_train = n - n_val - n_test
train_ds, val_ds, test_ds = random_split(active_ds, [n_train, n_val, n_test])

# DataLoaders — pin_memory + persistent_workers for maximum L4 throughput
_loader_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True
)
train_loader = DataLoader(train_ds, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)

print(f"\n✅ DataLoaders ready.")
print(f"   Train : {len(train_ds):,} samples | {len(train_loader)} batches")
print(f"   Val   : {len(val_ds):,}  samples | {len(val_loader)}  batches")
print(f"   Test  : {len(test_ds):,}  samples | {len(test_loader)}  batches")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Phase Generator — V4.0 Master Script
#   Phase 1 : Siamese training loop  (5 epochs, AntisymmetricLoss)
#   Phase 2 : Test-set benchmarking  (RMSE, Pearson r)
#   Phase 3 : Matplotlib visualisation  → holognn_final_metrics.png
# ══════════════════════════════════════════════════════════════════════════════

from scipy.stats import pearsonr
import matplotlib
matplotlib.use("Agg")   # headless-safe for Vertex AI
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


# ════════════════════════════════════════════════════════
# Architecture — inlined from src/  (no local package needed)
# ════════════════════════════════════════════════════════

# ── V2.0: Attention-based graph builder ──────────────────────────────────────
def build_attention_graph(avg_attention: torch.Tensor, threshold: float = 0.05) -> torch.Tensor:
    """Build edge_index from a (L, L) ESM-2 averaged attention map."""
    adj        = avg_attention + avg_attention.t()
    rows, cols = torch.where(adj > threshold)
    mask       = rows != cols
    return torch.stack([rows[mask], cols[mask]], dim=0)


ESM2_HIDDEN_DIM  = 320
MECH_FEATURE_DIM = 3
GAT_IN_CHANNELS  = ESM2_HIDDEN_DIM + MECH_FEATURE_DIM   # 323


# ── V3.0 + V2.0: Backbone ─────────────────────────────────────────────────────
class HoloGNNBackbone(nn.Module):
    """
    ESM-2 → Mechanistic Injection → Attention Graph → GATConv × 2 → Mean Pool.
    Inlined from src/backbone.py (V3.0).
    """
    def __init__(self, output_dim: int = 320):
        super().__init__()
        self.esm = EsmModel.from_pretrained(
            "facebook/esm2_t6_8M_UR50D", output_attentions=True
        )
        if _PYGEO:
            self.gat1 = GATConv(GAT_IN_CHANNELS, GAT_IN_CHANNELS, heads=4, concat=False)
            self.gat2 = GATConv(GAT_IN_CHANNELS, output_dim,      heads=4, concat=False)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, mechanistic_features, edge_index=None):
        B = input_ids.size(0)

        # Step 1 — ESM-2
        esm_out  = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        node_emb = esm_out.last_hidden_state        # (B, L, 320)
        L        = node_emb.size(1)

        # Step 2 — V3.0: concatenate mechanistic features
        node_emb = torch.cat([node_emb, mechanistic_features], dim=-1)  # (B, L, 323)

        # Step 3 — V2.0: build attention graph
        if edge_index is None and _PYGEO:
            last_attn   = esm_out.attentions[-1]               # (B, heads, L, L)
            avg_attn    = torch.mean(last_attn, dim=1)         # (B, L, L)
            batch_avg   = torch.mean(avg_attn,  dim=0)         # (L, L)
            edge_index  = build_attention_graph(batch_avg).to(input_ids.device)

        # Step 4 — GATConv message passing
        x = node_emb.view(-1, node_emb.size(-1))              # (B*L, 323)
        if edge_index is not None and _PYGEO:
            x = self.relu(self.gat1(x, edge_index))
            x = self.gat2(x, edge_index)

        # Step 5 — Mean pool → graph embedding
        x_r       = x.view(B, L, -1)
        graph_emb = torch.mean(x_r, dim=1)                    # (B, output_dim)
        return x_r, graph_emb


# ── Prediction heads  (from src/heads.py) ────────────────────────────────────
class ProteomicsHead(nn.Module):
    def __init__(self, input_dim=320):
        super().__init__()
        self.reg = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, z): return self.reg(z)

class SiameseStabilityHead(nn.Module):
    def __init__(self, input_dim=320):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, z_wt, z_mt): return self.mlp(z_mt - z_wt)

class EnsembleIDRHead(nn.Module):
    def __init__(self, input_dim=320):
        super().__init__()
        self.mu    = nn.Linear(input_dim, 1)
        self.sigma = nn.Linear(input_dim, 1)
        self.sp    = nn.Softplus()
    def forward(self, z): return self.mu(z), self.sp(self.sigma(z))


# ── V4.0: Full HoloGNN  (from src/full_model.py) ─────────────────────────────
class _DataBatch:
    """Lightweight namespace for backbone inputs."""
    __slots__ = ("input_ids", "mask", "mechanistic_features", "edge_index")

class HoloGNN(nn.Module):
    """
    V4.0 HoloGNN — Siamese Antisymmetric Multi-Task Predictor.
    task='idr'  → expects data = (data_wt, data_mt) tuple.
    """
    def __init__(self):
        super().__init__()
        self.backbone        = HoloGNNBackbone(output_dim=320)
        self.proteomics_head = ProteomicsHead(320)
        self.siamese_head    = SiameseStabilityHead(320)
        self.idr_head        = EnsembleIDRHead(320)

    def _encode(self, data):
        _, z = self.backbone(
            data.input_ids, data.mask,
            data.mechanistic_features, data.edge_index
        )
        return z

    def forward(self, data, task="proteomics"):
        if task == "proteomics":
            return self.proteomics_head(self._encode(data))
        if task == "idr":
            # V4.0: unpack (data_wt, data_mt) Siamese pair
            data_wt, data_mt = data
            z_wt = self._encode(data_wt)
            z_mt = self._encode(data_mt)
            dG_fwd = self.siamese_head(z_wt, z_mt)   # WT → MT
            dG_rev = self.siamese_head(z_mt, z_wt)   # MT → WT (antisymmetry check)
            return dG_fwd, dG_rev
        return self._encode(data)


# ── V4.0: AntisymmetricLoss  (from src/loss.py) ───────────────────────────────
class AntisymmetricLoss(nn.Module):
    """
    L = alpha*(dG_wt_to_mt + dG_mt_to_wt)^2 + (dG_pred - dG_exp)^2
         ─────────────────────────────────────   ─────────────────────
              Antisymmetry Term                     Fidelity Term
    """
    def __init__(self, alpha: float = 1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, dG_fwd, dG_rev, dG_exp):
        fwd = dG_fwd.squeeze(-1)
        rev = dG_rev.squeeze(-1)
        antisymmetry = (fwd + rev) ** 2
        fidelity     = (fwd - dG_exp) ** 2
        loss = torch.mean(self.alpha * antisymmetry + fidelity)
        comps = {
            "antisymmetry": torch.mean(antisymmetry).item(),
            "fidelity":     torch.mean(fidelity).item(),
        }
        return loss, comps


# ════════════════════════════════════════════════════════════════════
# Siamese Batch Builder
# ════════════════════════════════════════════════════════════════════
# MegaScaleDataset contains individual sequences, not WT/MT pairs.
# For the Siamese pass we treat consecutive batch halves as pseudo-pairs:
#   first  BATCH_SIZE//2 samples → wild-type  (data_wt)
#   second BATCH_SIZE//2 samples → mutant     (data_mt)
# This simulates the WT/MT split while remaining compatible with a
# single-dataset training loop. A production run would pair by gene ID.

def make_siamese_pair(batch, device):
    """Split a collated batch into (data_wt, data_mt) DataBatch objects."""
    half = batch["input_ids"].size(0) // 2
    if half == 0:
        raise ValueError("Batch too small to split into Siamese pairs (need >= 2).")

    def _make(sl):
        d               = _DataBatch()
        d.input_ids     = batch["input_ids"][sl].to(device, non_blocking=True)
        d.mask          = batch["attention_mask"][sl].to(device, non_blocking=True)
        d.mechanistic_features = batch["mechanistic_features"][sl].to(device, non_blocking=True)
        d.edge_index    = None   # Built dynamically from attention in backbone
        return d

    data_wt = _make(slice(None, half))
    data_mt = _make(slice(half, half * 2))
    labels  = batch["label"][:half].to(device, non_blocking=True)
    return data_wt, data_mt, labels


# ════════════════════════════════════════════════════════════════════
# Model + Optimiser + Scheduler
# ════════════════════════════════════════════════════════════════════
print(f"\nInitialising HoloGNN V4.0 on {device} ...")
model     = HoloGNN().to(device)
criterion = AntisymmetricLoss(alpha=ALPHA_ASYM)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE * 10,
    steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1
)
_total_p = sum(p.numel() for p in model.parameters())
print(f"   Total parameters: {_total_p:,}")


# ════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────
# PHASE 1 — TRAINING  (5 epochs · AntisymmetricLoss)
# ─────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("  PHASE 1 — TRAINING")
print("  Task   : Siamese ΔΔG regression (task='idr')")
print(f"  Loss   : AntisymmetricLoss (alpha={ALPHA_ASYM})")
print(f"  Epochs : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  Device : {device}")
print("═" * 62)

history = {"train_total": [], "train_fidelity": [], "train_antisymmetry": [],
           "val_total":   [], "val_fidelity":   [], "val_antisymmetry":   []}
best_val   = float("inf")
wall_start = time.time()


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, tot_fid, tot_asym, n_batches = 0.0, 0.0, 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for batch in tqdm(loader, leave=False):
            try:
                data_wt, data_mt, labels = make_siamese_pair(batch, device)
            except ValueError:
                continue   # skip under-sized tail batch

            if train: optimizer.zero_grad()

            dG_fwd, dG_rev = model((data_wt, data_mt), task="idr")
            loss, comps    = criterion(dG_fwd, dG_rev, labels)

            if train:
                loss.backward()
                optimizer.step()
                scheduler.step()

            tot_loss += loss.item()
            tot_fid  += comps["fidelity"]
            tot_asym += comps["antisymmetry"]
            n_batches += 1

    n = max(n_batches, 1)
    return tot_loss / n, tot_fid / n, tot_asym / n


for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_fid, tr_asym = run_epoch(train_loader, train=True)
    va_loss, va_fid, va_asym = run_epoch(val_loader,   train=False)
    elapsed = time.time() - t0

    history["train_total"].append(tr_loss)
    history["train_fidelity"].append(tr_fid)
    history["train_antisymmetry"].append(tr_asym)
    history["val_total"].append(va_loss)
    history["val_fidelity"].append(va_fid)
    history["val_antisymmetry"].append(va_asym)

    flag = ""
    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        flag = " ✅ saved"

    print(
        f"Epoch {epoch:>2}/{EPOCHS}  "
        f"total={tr_loss:.4f}  fid={tr_fid:.4f}  asym={tr_asym:.4f}  "
        f"| val_total={va_loss:.4f}  val_fid={va_fid:.4f}  "
        f"val_asym={va_asym:.4f}  [{elapsed:.0f}s]{flag}"
    )

print(f"\n✅ Phase 1 complete in {(time.time()-wall_start)/3600:.2f} h.")
print(f"   Best val loss: {best_val:.4f} → {CHECKPOINT_PATH}")


# ════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────
# PHASE 2 — BENCHMARKING  (RMSE, Pearson r on held-out test set)
# ─────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("  PHASE 2 — BENCHMARKING")
print("═" * 62)

# Reload best weights
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
asym_violations       = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        try:
            data_wt, data_mt, labels = make_siamese_pair(batch, device)
        except ValueError:
            continue

        dG_fwd, dG_rev = model((data_wt, data_mt), task="idr")
        fwd = dG_fwd.squeeze(-1).cpu()
        rev = dG_rev.squeeze(-1).cpu()
        lbl = labels.cpu()

        all_preds.extend(fwd.tolist())
        all_labels.extend(lbl.tolist())
        # Track antisymmetry violation magnitude |dG_fwd + dG_rev|
        asym_violations.extend((fwd + rev).abs().tolist())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
asym_arr   = np.array(asym_violations)

rmse    = float(np.sqrt(np.mean((all_preds - all_labels) ** 2)))
pearson = float(pearsonr(all_preds, all_labels)[0])
mae     = float(np.mean(np.abs(all_preds - all_labels)))
mean_asym_viol = float(asym_arr.mean())

print(f"\n  Test-set Benchmarks ({len(all_labels):,} pairs)")
print(f"  ─────────────────────────────────────")
print(f"  RMSE                  : {rmse:.4f} kcal/mol")
print(f"  MAE                   : {mae:.4f} kcal/mol")
print(f"  Pearson r             : {pearson:.4f}")
print(f"  Mean Antisymmetry Viol: {mean_asym_viol:.4f}  (lower → more physical)")


# ════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────
# PHASE 3 — VISUALISATION  → holognn_final_metrics.png
# ─────────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("  PHASE 3 — VISUALISATION")
print("═" * 62)

plt.style.use("seaborn-v0_8-darkgrid")
ep_range = range(1, EPOCHS + 1)

fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    "Holo-GNN V4.0 — Final Training & Evaluation Metrics",
    fontsize=16, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# ── Plot 1: Total Loss (train vs val) ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ep_range, history["train_total"], marker="o", label="Train")
ax1.plot(ep_range, history["val_total"],   marker="s", label="Val", linestyle="--")
ax1.set_title("Total AntisymmetricLoss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend()

# ── Plot 2: Fidelity Term ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep_range, history["train_fidelity"], marker="o", color="steelblue",  label="Train")
ax2.plot(ep_range, history["val_fidelity"],   marker="s", color="dodgerblue", label="Val", linestyle="--")
ax2.set_title("Fidelity Term  (pred − exp)²"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("MSE")
ax2.legend()

# ── Plot 3: Antisymmetry Term ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ep_range, history["train_antisymmetry"], marker="o", color="tomato",      label="Train")
ax3.plot(ep_range, history["val_antisymmetry"],   marker="s", color="lightsalmon", label="Val", linestyle="--")
ax3.set_title("Antisymmetry Term  (fwd + rev)²"); ax3.set_xlabel("Epoch"); ax3.set_ylabel("Penalty")
ax3.legend()

# ── Plot 4: Predicted vs. Experimental ΔΔG scatter ──────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.scatter(all_labels, all_preds, alpha=0.35, s=6, color="mediumseagreen", label="Test pairs")
lims = [min(all_labels.min(), all_preds.min()), max(all_labels.max(), all_preds.max())]
ax4.plot(lims, lims, "k--", linewidth=1, label="y = x (ideal)")
ax4.set_xlabel("Experimental ΔΔG (kcal/mol)")
ax4.set_ylabel("Predicted ΔΔG (kcal/mol)")
ax4.set_title(
    f"Predicted vs. Experimental ΔΔG\n"
    f"RMSE={rmse:.4f}  MAE={mae:.4f}  Pearson r={pearson:.4f}"
)
ax4.legend(markerscale=3)

# ── Plot 5: Antisymmetry Violation Distribution ──────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(asym_arr, bins=50, color="mediumpurple", edgecolor="white", linewidth=0.4)
ax5.axvline(mean_asym_viol, color="red", linestyle="--",
            label=f"Mean = {mean_asym_viol:.4f}")
ax5.set_title("|dG_fwd + dG_rev| Distribution\n(Antisymmetry Violation)")
ax5.set_xlabel("|Violation| (kcal/mol)"); ax5.set_ylabel("Count")
ax5.legend()

plt.savefig("holognn_final_metrics.png", dpi=150, bbox_inches="tight")
print("\n✅ Phase 3 complete — holognn_final_metrics.png saved.")
plt.show()

print("\n" + "═" * 62)
print("  ALL PHASES COMPLETE")
print(f"  Checkpoint : {CHECKPOINT_PATH}")
print(f"  Metrics    : holognn_final_metrics.png")
print(f"  RMSE       : {rmse:.4f} kcal/mol")
print(f"  Pearson r  : {pearson:.4f}")
print("═" * 62)